# BKQuiz · Qwen7B FAST · phiên tối đa 55 phút
Sinh theo batch, kiểm tra JSON/trích dẫn/trùng lặp trên CPU. Chỉ giải lại hai thứ tự và critic
cho câu rủi ro hoặc mẫu ngẫu nhiên 10%. Câu chưa được model kiểm tra lại vẫn phải duyệt thủ công.

Mặc định xử lý gói full, batch 2 và giới hạn 55 phút kể từ ô cấu hình, gồm cài thư viện/tải model.
Không cam kết xong 1.994 câu trong một phiên. Chạy lại để tiếp tục cache của bản FAST.
Cache và đầu ra bản strict cũ vẫn giữ nguyên, nhập kết quả cũ trên web vẫn được hỗ trợ.
Chọn Runtime → Change runtime type → T4 GPU (hoặc GPU CUDA ≥14 GiB).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile, os, time
BASE=Path('/content/drive/MyDrive/BKQuiz_CPT/SFT_PEDAGOGY_QWEN7B_FAST_20261025')
ASSETS={'runtime_sha256': '994a5b6da16064fb73df23e9910c239720844c1dfdc3b15ac48d57918ceadd87', 'pilot_sha256': '01a1686abd23bbafa87cd01eeb6223c806b5b4717c464231473ceffb80f8c619', 'full_sha256': 'bf6e86e1f5ec37e1a98d6e4443ad7d509fbe1c3b714711187fd18215e67ca923'}
PACKET_KIND='full'  # 'pilot' =63 jobs, 'full' =1994 jobs
MAX_JOBS=None  # 3 để chạy thử
BATCH_SIZE=2  # T4: bắt đầu2, tự giảm nếu OOM
SESSION_MINUTES=55
RETRY_FAILED=False
assert 1<=BATCH_SIZE<=8 and 1<=SESSION_MINUTES<=55
SESSION_STARTED=time.monotonic()
DEADLINE=SESSION_STARTED+SESSION_MINUTES*60
def remaining():
    value=DEADLINE-time.monotonic()
    if value<=30: raise TimeoutError('Hết ngân sách phiên; chạy lại để tiếp tục')
    return value
def budget_run(command, **kwargs):
    return subprocess.run(command, timeout=remaining(), **kwargs)
assert BASE.is_dir(), f'Chưa thấy thư mục {BASE}'
print('Tối đa', SESSION_MINUTES, 'phút; model audit theo rủi ro +10% mẫu')

In [ ]:
# Kiểm tra tệp trước khi cài đặt hoặc thực thi mã từ gói đã đóng băng.
def checked_file(name, expected):
    path = BASE / name
    with path.open('rb') as stream:
        actual = hashlib.file_digest(stream, 'sha256').hexdigest()
    if actual != expected:
        raise ValueError(f'SHA256 không khớp: {name}; tải lại đúng gói, không sửa hash trong notebook')
    return path
runtime_zip = checked_file('teacher_runtime.zip', ASSETS['runtime_sha256'])
input_zip = checked_file('teacher_' + PACKET_KIND + '.zip', ASSETS[PACKET_KIND + '_sha256'])
WORK = Path('/content/bkquiz_pedagogy_fast_runtime')
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(runtime_zip) as archive:
    expected = {'pedagogy_core.py','teacher_runtime.py','teacher_fast_runtime.py','requirements.txt','runtime_manifest.json'}
    if set(archive.namelist()) != expected or any(i.file_size > 1024*1024 for i in archive.infolist()):
        raise ValueError('Gói runtime không hợp lệ')
    raw = {n: archive.read(n) for n in expected}
    manifest = json.loads(raw['runtime_manifest.json'])
    for name, wanted in manifest['files_sha256'].items():
        if hashlib.sha256(raw[name]).hexdigest() != wanted:
            raise ValueError('Runtime file bị thay đổi: ' + name)
    for name, body in raw.items():
        (WORK / name).write_bytes(body)
print('Verified runtime/input:', runtime_zip.name, input_zip.name)

In [ ]:
# Môi trường riêng tránh ghi đè thư viện của kernel Colab đang chạy.
VENV = Path('/content/bkquiz_pedagogy_venv')
PYTHON = VENV / 'bin/python'
if not PYTHON.exists():
    budget_run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
# Reuse Colab's working CUDA torch (including Python3.13 runtimes),
# rather than downloading/replacing it on every run.
budget_run([str(PYTHON),'-c','import torch; assert torch.cuda.is_available(), \"Select a working Colab GPU runtime\"; print(\"CUDA torch\",torch.__version__)'],check=True)
# Không dùng vision/audio trong teacher văn bản này. Bỏ các bản hệ thống
# khỏi kiểm tra optional dependency để tránh ABI mismatch với torch riêng.
guard = WORK / 'sitecustomize.py'
guard.write_text("import sys\nimport importlib.util\n_original = importlib.util.find_spec\ndef _find(name, *args, **kwargs):\n    if name.split('.')[0] in {'torchvision','torchaudio','librosa'}: return None\n    return _original(name, *args, **kwargs)\nimportlib.util.find_spec = _find\n", encoding='utf-8')
budget_run([str(PYTHON), '-m', 'pip', 'install', '--quiet', '-r', str(WORK/'requirements.txt')], check=True)
env = dict(os.environ, PYTHONPATH=str(WORK), TOKENIZERS_PARALLELISM='false')
budget_run([str(PYTHON), '-c', 'import torch, transformers, bitsandbytes; print(torch.__version__, transformers.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_properties(0))'], env=env, check=True)

In [ ]:
OUTPUT=BASE/'runs'/PACKET_KIND
OUTPUT.mkdir(parents=True,exist_ok=True)
# Model load is included in this remaining budget. Reserve90s for ZIP/download.
minutes=min(50,max(.01,(remaining()-90)/60))
command=[str(PYTHON),'-u',str(WORK/'teacher_fast_runtime.py'),'--input',str(input_zip),'--out',str(OUTPUT),
         '--batch-size',str(BATCH_SIZE),'--minutes',str(minutes)]
if MAX_JOBS is not None: command+=['--max-jobs',str(MAX_JOBS)]
if RETRY_FAILED: command+=['--retry-failed']
try:
    budget_run(command,env=env,cwd=str(WORK),check=True)
except subprocess.TimeoutExpired:
    print('Watchdog đã dừng phiên. Cache mỗi câu được giữ trên Drive; chạy lại để tiếp tục.')

In [ ]:
from google.colab import files
candidates=list(OUTPUT.glob('pedagogy_results_*.zip'))+list(OUTPUT.glob('pedagogy_checkpoint.zip'))
# Do not silently download an old run as if this session succeeded.
recent=[p for p in candidates if p.stat().st_mtime>=time.time()-(time.monotonic()-SESSION_STARTED)-5]
if recent:
    result=max(recent,key=lambda p:p.stat().st_mtime)
    print('Kết quả thật đã hoàn thành:',result)
    print('Nhập ZIP tại http://127.0.0.1:8765/sft/pedagogy/ rồi duyệt từng câu.')
    files.download(str(result))
else:
    print('Phiên này chưa có ZIP mới. Cache còn trên Drive; kiểm tra log và chạy lại để tiếp tục.')
report=OUTPUT/'SESSION_REPORT.json'
if report.is_file(): print(report.read_text())

## Tiếp tục và duyệt
Chạy lại Run all với cùng gói để tiếp tục. Cache đã hoàn thành không sinh lại; draft đang chờ kiểm tra
được giữ riêng và chưa xuất. Mỗi30giây có log sinh token; sau mỗi batch có progress và checkpoint ZIP.
Câu rủi ro: multiple-select, ghép môn liên quan, retrieval yếu hoặc câu hỏi tính toán/có số.
10% mẫu ngẫu nhiên được chọn ổn định theo job ID, không thay đổi khi chạy lại.

Web phân biệt **chỉ kiểm tra CPU** và **đã kiểm tra thêm bằng model**. Cả hai đều chờ người dùng duyệt,
không có xác nhận chuyên gia tự động. Đáp án mâu thuẫn nguồn/trích dẫn bịa vẫn bị chặn.
Chỉ các câu bạn chấp nhận được xuất SFT; tỷ lệ train vẫn ưu tiên≥80% câu mới từ lý thuyết.